Train Item-Level Models - Live Production Data
Trains weekly (LightGBM) and monthly (Prophet) models per item (SKU), using hybrid model vs baseline selection, reconciled to the overall forecast.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold, append_json_history, save_history_as_excel
import pandas as pd
import json
import io
import datetime
import lightgbm as lgb
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

FORECAST_BASE = "live/battery"

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

def compute_working_days(period_start, period_end, holiday_dates_set):
    all_days = pd.date_range(period_start, period_end)
    return sum(1 for d in all_days if d.weekday() != 6 and d.date() not in holiday_dates_set)

holiday_calendar = read_gold(blob_service, f"{FORECAST_BASE}/data/reference/holiday_calendar.parquet")
holiday_calendar["date"] = pd.to_datetime(holiday_calendar["date"])
holiday_dates_set = set(holiday_calendar["date"].dt.date)

Load weekly gold

In [0]:
gold_item_weekly = read_gold(blob_service, f"{FORECAST_BASE}/data/phase5_item_weekly_live.parquet")
gold_item_weekly["week_start"] = pd.to_datetime(gold_item_weekly["week_start"])
gold_item_weekly["item_group"] = gold_item_weekly["item_group"].astype("category")

items = gold_item_weekly["item_group"].cat.categories.tolist()
print(f"Total item groups: {len(items)}")

Weekly: train/test split, train, evaluate per item

In [0]:
feature_cols_item = ["week_of_year", "month", "contains_month_end", "rate_lag_4w", "rate_rolling_avg_4w", "item_group"]
target_col_rate = "units_per_working_day"

model_data_item = gold_item_weekly.dropna(subset=["rate_lag_4w", "rate_rolling_avg_4w", target_col_rate]).copy()
model_data_item = model_data_item.sort_values("week_start")

split_idx = int(len(model_data_item) * 0.8)
train_item = model_data_item.iloc[:split_idx]
test_item = model_data_item.iloc[split_idx:]

X_train_i, y_train_i = train_item[feature_cols_item], train_item[target_col_rate]
X_test_i, y_test_i = test_item[feature_cols_item], test_item[target_col_rate]

model_item = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
model_item.fit(X_train_i, y_train_i, categorical_feature=["item_group"])

rate_preds_i = model_item.predict(X_test_i)
test_item_results = test_item.copy()
test_item_results["rate_prediction"] = rate_preds_i
test_item_results["total_prediction"] = test_item_results["rate_prediction"] * test_item_results["working_days"]
test_item_results["baseline_total"] = test_item_results["rate_rolling_avg_4w"] * test_item_results["working_days"]

item_weekly_method = {}
for item in test_item_results["item_group"].unique():
    subset = test_item_results[test_item_results["item_group"] == item]
    model_wape = wape(subset["total_units_sold"], subset["total_prediction"])
    baseline_wape = wape(subset["total_units_sold"], subset["baseline_total"])
    item_weekly_method[item] = "model" if model_wape < baseline_wape else "baseline"

model_wins = sum(1 for v in item_weekly_method.values() if v == "model")
print(f"Weekly: {model_wins}/{len(item_weekly_method)} items favor the model")

Weekly: retrain on all data, predict next week per item

In [0]:
final_item_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
X_all_i = model_data_item[feature_cols_item]
y_all_i = model_data_item[target_col_rate]
final_item_model.fit(X_all_i, y_all_i, categorical_feature=["item_group"])

last_week_start = gold_item_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)
next_week_working_days = max(compute_working_days(next_week_start, next_week_end, holiday_dates_set), 1)

item_weekly_predictions = {}
for item in items:
    item_hist = gold_item_weekly[gold_item_weekly["item_group"] == item]

    if item_weekly_method.get(item) == "model":
        lag_val = item_hist[item_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["units_per_working_day"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "rate_lag_4w": lag_val,
            "rate_rolling_avg_4w": item_hist["units_per_working_day"].tail(4).mean(),
            "item_group": item,
        }])
        row["item_group"] = row["item_group"].astype("category")
        predicted_rate = final_item_model.predict(row[feature_cols_item])[0]
    else:
        predicted_rate = item_hist["units_per_working_day"].tail(4).mean()

    item_weekly_predictions[item] = max(predicted_rate * next_week_working_days, 0)

print(pd.Series(item_weekly_predictions).sort_values(ascending=False))

Reconcile weekly to overall active forecast

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_weekly_overall = json.loads(stream)

overall_target = next(
    (w["predicted_units"] for w in active_weekly_overall if w["week_start"] == next_week_start.strftime("%Y-%m-%d")),
    sum(item_weekly_predictions.values())
)

item_sum = sum(item_weekly_predictions.values())
item_weekly_reconciled = {
    item: (val / item_sum) * overall_target if item_sum > 0 else 0
    for item, val in item_weekly_predictions.items()
}

print(f"Overall target: {overall_target}   Item sum (raw): {item_sum:.0f}   Item sum (reconciled): {sum(item_weekly_reconciled.values()):.0f}")

Load monthly gold, per-item Prophet with hybrid selection

In [0]:
gold_item_monthly = read_gold(blob_service, f"{FORECAST_BASE}/data/phase5_item_monthly_live.parquet")
gold_item_monthly["month_start"] = pd.to_datetime(gold_item_monthly["month_start"])

global_last_month = gold_item_monthly["month_start"].max()

item_monthly_method = {}
item_monthly_forecast = {}

for item in items:
    item_df = gold_item_monthly[gold_item_monthly["item_group"] == item][["month_start", "total_units_sold"]]
    item_df = item_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if len(item_df) < 15 or item_df["y"].tail(12).sum() == 0:
        method = "baseline"
    else:
        train_i = item_df.iloc[:-3]
        test_i = item_df.iloc[-3:]
        try:
            m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
            m_test.fit(train_i)
            future_test = m_test.make_future_dataframe(periods=3, freq="MS")
            forecast_test = m_test.predict(future_test)
            test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

            model_wape = wape(test_i["y"].values, test_preds)
            naive_pred = train_i["y"].tail(3).mean()
            baseline_wape = wape(test_i["y"].values, [naive_pred] * 3)
            method = "model" if model_wape < baseline_wape else "baseline"
        except Exception:
            method = "baseline"

    item_monthly_method[item] = method

    if method == "model":
        m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
        m_final.fit(item_df)
        future_final = m_final.make_future_dataframe(periods=3, freq="MS")
        forecast_final = m_final.predict(future_final)
        preds = forecast_final.tail(3)["yhat"].clip(lower=0).values
        month_labels = forecast_final.tail(3)["ds"].dt.strftime("%Y-%m-%d").tolist()
    else:
        flat_value = max(item_df["y"].tail(3).mean(), 0) if len(item_df) > 0 else 0.0
        preds = [flat_value] * 3
        last_month = item_df["ds"].max() if len(item_df) > 0 else global_last_month
        month_labels = [(last_month + pd.DateOffset(months=i)).strftime("%Y-%m-01") for i in range(1, 4)]

    item_monthly_forecast[item] = {"months": month_labels, "values": [float(p) for p in preds]}

model_wins_m = sum(1 for v in item_monthly_method.values() if v == "model")
print(f"Monthly: {model_wins_m}/{len(item_monthly_method)} items favor the model")

Reconcile monthly to overall active forecast, per month

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_monthly_overall = json.loads(stream)

overall_by_month = {m["month_start"]: m["predicted_units"] for m in active_monthly_overall}

item_monthly_reconciled = {item: {"months": [], "values": []} for item in items}

sample_months = item_monthly_forecast[items[0]]["months"]
for i, month_label in enumerate(sample_months):
    month_item_sum = sum(item_monthly_forecast[item]["values"][i] for item in items)
    overall_target = overall_by_month.get(month_label, month_item_sum)

    for item in items:
        raw_val = item_monthly_forecast[item]["values"][i]
        reconciled_val = (raw_val / month_item_sum) * overall_target if month_item_sum > 0 else 0
        item_monthly_reconciled[item]["months"].append(month_label)
        item_monthly_reconciled[item]["values"].append(reconciled_val)

print(f"Sum check (month 1): {sum(item_monthly_reconciled[item]['values'][0] for item in items):.0f}  target: {overall_by_month.get(sample_months[0])}")

Save active + history + Excel (weekly)

In [0]:
today_str = datetime.date.today().isoformat()
today = pd.Timestamp(datetime.date.today())

weekly_records = [
    {
        "generated_date": today_str,
        "week_start": next_week_start.strftime("%Y-%m-%d"),
        "week_end": (next_week_start + pd.Timedelta(days=6)).strftime("%Y-%m-%d"),
        "item_no": item,
        "predicted_units": round(float(val))
    }
    for item, val in item_weekly_reconciled.items()
]

weekly_history = append_json_history(blob_service, weekly_records, f"{FORECAST_BASE}/forecasts/history/item_weekly_forecast_history.json")

weekly_df = pd.DataFrame(weekly_history)
weekly_df["week_end"] = pd.to_datetime(weekly_df["week_end"])
weekly_df["generated_date"] = pd.to_datetime(weekly_df["generated_date"])

active_weekly_i = weekly_df[weekly_df["week_end"] >= today]
active_weekly_i = active_weekly_i.sort_values("generated_date").drop_duplicates(subset=["week_start", "item_no"], keep="last")
active_weekly_i = active_weekly_i.sort_values(["week_start", "item_no"])

active_weekly_i_records = active_weekly_i.to_dict(orient="records")
for r in active_weekly_i_records:
    r["week_end"] = r["week_end"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/item_weekly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_weekly_i_records, indent=2), overwrite=True)
print(f"Active item weekly: {len(active_weekly_i_records)} records")

count = save_history_as_excel(blob_service, weekly_history, f"{FORECAST_BASE}/forecasts/history/item_weekly_forecast_history.xlsx")
print(f"Saved item weekly Excel: {count} rows")

Save active + history + Excel (monthly)

In [0]:
monthly_records = []
for item, data in item_monthly_reconciled.items():
    for month_label, val in zip(data["months"], data["values"]):
        monthly_records.append({
            "generated_date": today_str,
            "month_start": month_label,
            "item_no": item,
            "predicted_units": round(float(val))
        })

monthly_history = append_json_history(blob_service, monthly_records, f"{FORECAST_BASE}/forecasts/history/item_monthly_forecast_history.json")

monthly_df = pd.DataFrame(monthly_history)
monthly_df["month_start"] = pd.to_datetime(monthly_df["month_start"])
monthly_df["generated_date"] = pd.to_datetime(monthly_df["generated_date"])
monthly_df["month_end"] = monthly_df["month_start"] + pd.offsets.MonthEnd(0)

active_monthly_i = monthly_df[monthly_df["month_end"] >= today]
active_monthly_i = active_monthly_i.sort_values("generated_date").drop_duplicates(subset=["month_start", "item_no"], keep="last")
active_monthly_i = active_monthly_i.sort_values(["month_start", "item_no"])

active_monthly_i_records = active_monthly_i.drop(columns=["month_end"]).to_dict(orient="records")
for r in active_monthly_i_records:
    r["month_start"] = r["month_start"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{FORECAST_BASE}/forecasts/active/item_monthly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_monthly_i_records, indent=2), overwrite=True)
print(f"Active item monthly: {len(active_monthly_i_records)} records")

count = save_history_as_excel(blob_service, monthly_history, f"{FORECAST_BASE}/forecasts/history/item_monthly_forecast_history.xlsx")
print(f"Saved item monthly Excel: {count} rows")